# B_S3 — Unified Permutation Test

Runs 500-permutation tests for **all metrics** on the combined dataset
(main + expanded null). Stratified case selection controls pipeline scale:

| RUN_MODE | true_null | variance_only | mean_only | mean+variance | ~Total |
|----------|-----------|---------------|-----------|---------------|--------|
| `quick` | 1,000 | 1,000 | 1,000 | 1,000 | **4K** |
| `quick_medium` | 2,000 | 2,000 | 2,000 | 2,000 | **8K** |
| `medium` | ALL (4,032) | ALL (2,976) | 5,000 | 5,000 | **~17K** |
| `full` | ALL | ALL | ALL | ALL | **~98K** |

Five phases, each with checkpointing:

| Phase | Metrics | Speed | Cases |
|-------|---------|-------|-------|
| 1 | Core 4 + slopes + bins + covariance | Vectorized, fast | All selected |
| 2 | Distance covariance | O(n²) per perm, slow | All selected |
| 3 | Distribution (KS, Wasserstein) | Moderate | All selected |
| 4 | MINE (MIC/MAS/MEV/MCN) | Subset mode, slow | Slow-phase subset |
| 5 | LOWESS R² | Subset mode, very slow | Slow-phase subset |

Joint test: max-Z across all metrics → p-value → classification.

Output: `output/S3/permutation_all.parquet` + phase checkpoints

In [13]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata, ks_2samp, wasserstein_distance, pearsonr
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess

try:
    from minepy import MINE as MINEObj
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print('minepy not installed — Phase 4 (MINE) will be skipped.')

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

warnings.filterwarnings('ignore')

In [14]:
# ── Configuration ──
S1_DIR    = Path('output/S1')
OUT_DIR   = Path('output/S3')
CKPT_DIR  = OUT_DIR / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

N_PERM    = 100
SEED_BASE = 42_000_000

# ── Run mode: controls which cases enter the ENTIRE pipeline ──
# 'quick'        — 1,000 per category (~4K total), fast iteration
# 'quick_medium' — 2,000 per category (~8K total)
# 'medium'       — True Null ALL + Variance-only ALL + 5K mean_only + 5K mean+variance (~17K)
# 'full'         — all cases (~98K)
RUN_MODE = 'medium'

CATEGORY_BUDGETS = {
    'quick':        {'true_null': 1000, 'variance_only': 1000, 'mean_only': 1000, 'mean+variance': 1000},
    'quick_medium': {'true_null': 2000, 'variance_only': 2000, 'mean_only': 2000, 'mean+variance': 2000},
    'medium':       {'true_null': 'all', 'variance_only': 'all', 'mean_only': 5000, 'mean+variance': 5000},
    'full':         {'true_null': 'all', 'variance_only': 'all', 'mean_only': 'all', 'mean+variance': 'all'},
}

# Phases 4 (MINE) and 5 (LOWESS) are extremely slow.
# Within the selected cases, these phases run on a further subset.
# Set to 'same' to use the same cases as phases 1-3, or an integer for a signal budget.
SLOW_PHASE_SIGNAL_BUDGET = 2000  # only affects phases 4-5; null/variance-only always included

# Metrics where lower observed value = stronger signal
LOWER_IS_SIGNAL = {'lowess_res_sd'}

## Load Combined Data

In [15]:
cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
pts_main = np.load(S1_DIR / 'scatter_points.npz')
pts_null = np.load(S1_DIR / 'null_expanded_points.npz')

cases_main['source'] = 'main'
cases_null['source'] = 'null_expanded'
offset = cases_main['case_id'].max()
cases_null['case_id'] = cases_null['case_id'] + offset

cases_all = pd.concat([cases_main, cases_null], ignore_index=True)
x_raw = np.concatenate([pts_main['x'], pts_null['x']], axis=0)
y_raw = np.concatenate([pts_main['y'], pts_null['y']], axis=0)
del pts_main, pts_null

# Assign categories
is_null = cases_all['family_id'] == 'Null'
is_const = cases_all['spread_pattern'] == 'constant'
cases_all['category'] = 'mean+variance'
cases_all.loc[is_null & is_const, 'category'] = 'true_null'
cases_all.loc[~is_null & is_const, 'category'] = 'mean_only'
cases_all.loc[is_null & ~is_const, 'category'] = 'variance_only'

print(f'Loaded: {len(cases_all):,} total cases')
print(cases_all['category'].value_counts().to_string())

# ── Stratified case selection (RUN_MODE) ──
budgets = CATEGORY_BUDGETS[RUN_MODE]
rng_sel = np.random.default_rng(42)
selected_idx = []

for cat in ['true_null', 'variance_only', 'mean_only', 'mean+variance']:
    cat_idx = np.where(cases_all['category'] == cat)[0]
    budget = budgets[cat]
    if budget == 'all' or budget >= len(cat_idx):
        selected_idx.append(cat_idx)
    else:
        cat_df = cases_all.iloc[cat_idx].copy()
        cat_df['_row_idx'] = cat_idx
        groups = cat_df.groupby(['family_id', 'snr'])
        per_group = max(1, budget // groups.ngroups)
        chosen = []
        for _, grp in groups:
            rows = grp['_row_idx'].values
            k = min(per_group, len(rows))
            chosen.extend(rng_sel.choice(rows, size=k, replace=False))
        if len(chosen) < budget:
            remaining = np.setdiff1d(cat_idx, chosen)
            extra = min(budget - len(chosen), len(remaining))
            if extra > 0:
                chosen.extend(rng_sel.choice(remaining, size=extra, replace=False))
        selected_idx.append(np.array(chosen[:budget]))

selected_idx = np.sort(np.concatenate(selected_idx))

cases_df = cases_all.iloc[selected_idx].reset_index(drop=True)
x_all = x_raw[selected_idx]
y_all = y_raw[selected_idx]
n_cases = len(cases_df)
del cases_all, x_raw, y_raw

print(f'\nRUN_MODE={RUN_MODE!r} → {n_cases:,} cases selected')
print(cases_df['category'].value_counts().to_string())

# ── Further subset for slow phases (4: MINE, 5: LOWESS) ──
if SLOW_PHASE_SIGNAL_BUDGET == 'same':
    subset_idx = np.arange(n_cases)
else:
    rng_sub = np.random.default_rng(99)
    null_mask = cases_df['category'].isin(['true_null', 'variance_only'])
    null_idx = np.where(null_mask)[0]

    signal_df = cases_df[~null_mask].copy()
    signal_df['_row_idx'] = np.where(~null_mask)[0]
    n_signal = len(signal_df)

    if SLOW_PHASE_SIGNAL_BUDGET >= n_signal:
        subset_idx = np.arange(n_cases)
    else:
        groups = signal_df.groupby(['family_id', 'snr'])
        per_group = max(1, SLOW_PHASE_SIGNAL_BUDGET // groups.ngroups)
        chosen = []
        for _, grp in groups:
            rows = grp['_row_idx'].values
            k = min(per_group, len(rows))
            chosen.extend(rng_sub.choice(rows, size=k, replace=False))
        if len(chosen) < SLOW_PHASE_SIGNAL_BUDGET:
            remaining = np.setdiff1d(signal_df['_row_idx'].values, chosen)
            extra = min(SLOW_PHASE_SIGNAL_BUDGET - len(chosen), len(remaining))
            if extra > 0:
                chosen.extend(rng_sub.choice(remaining, size=extra, replace=False))
        subset_idx = np.sort(np.concatenate([null_idx, np.array(chosen)]))

n_sub = len(subset_idx)
sub_cats = cases_df.iloc[subset_idx]['category'].value_counts()
print(f'\nSlow-phase subset → {n_sub:,} cases for MINE/LOWESS')
print(sub_cats.to_string())

Loaded: 121,056 total cases
category
mean+variance    85536
mean_only        28512
true_null         4032
variance_only     2976

RUN_MODE='medium' → 17,008 cases selected
category
mean_only        5000
mean+variance    5000
true_null        4032
variance_only    2976

Slow-phase subset → 9,008 cases for MINE/LOWESS
category
true_null        4032
variance_only    2976
mean+variance    1018
mean_only         982


## Helper Functions

In [16]:
def _generate_perms(n, n_perm, seed):
    rng = np.random.default_rng(seed)
    return np.array([rng.permutation(n) for _ in range(n_perm)])


def _double_center(a):
    D = squareform(pdist(a.reshape(-1, 1)))
    return D - D.mean(axis=0, keepdims=True) - D.mean(axis=1, keepdims=True) + D.mean()


def _precompute_bins(x, n_bins=10, min_count=5, bin_type='equal_width'):
    try:
        if bin_type == 'equal_width':
            bins = np.asarray(pd.cut(x, bins=n_bins, labels=False,
                                     include_lowest=True, duplicates='drop'), dtype=float)
        else:
            bins = np.asarray(pd.qcut(x, q=n_bins, labels=False, duplicates='drop'), dtype=float)
    except Exception:
        return None
    valid_ids = np.unique(bins[~np.isnan(bins)]).astype(int)
    masks, counts = [], []
    for b in valid_ids:
        m = bins == b
        if m.sum() >= min_count:
            masks.append(m)
            counts.append(m.sum())
    if len(masks) < 2:
        return None
    B_ind = np.array([m.astype(np.float64) for m in masks])
    return B_ind, np.array(counts, dtype=np.float64)


def _z_and_p(obs, null, direction=1):
    med = float(np.median(null))
    iqr = float(np.percentile(null, 75) - np.percentile(null, 25))
    if iqr < 1e-12:
        iqr = float(np.std(null)) * 1.35
    if iqr < 1e-12:
        return 0.0, np.zeros_like(null), med, iqr, 1.0
    if direction == -1:
        z_obs = (med - obs) / iqr
        z_null = (med - null) / iqr
    else:
        z_obs = (obs - med) / iqr
        z_null = (null - med) / iqr
    p = float(np.sum(null >= obs if direction == 1 else null <= obs) + 1) / (len(null) + 1)
    return float(z_obs), z_null, med, iqr, p

## Phase 1: Core + Vectorizable Metrics

|r|, |ρ|, η² (equal_width), dcor, covariance, slopes (raw/std), bin metrics, segment strength

In [17]:
def compute_phase1_case(x, y, perms):
    n = len(x)
    n_perm = len(perms)
    R = {}
    y_perms = y[perms]

    # ── |r| ──
    xc = x - x.mean(); yc = y - y.mean()
    sx = np.sqrt((xc**2).sum()); sy = np.sqrt((yc**2).sum())
    if sx > 0 and sy > 0:
        R['abs_pearson_r'] = (abs(float((xc*yc).sum()/(sx*sy))),
                              np.abs((xc * y_perms).sum(1) / (sx * np.sqrt(((y_perms - y_perms.mean(1, keepdims=True))**2).sum(1)))))
        R['abs_covariance'] = (abs(float((xc*yc).sum()/(n-1))),
                               np.abs((xc * (y_perms - y_perms.mean(1, keepdims=True))).sum(1)/(n-1)))
    else:
        R['abs_pearson_r'] = (0., np.zeros(n_perm))
        R['abs_covariance'] = (0., np.zeros(n_perm))

    # ── |ρ| ──
    xr = rankdata(x).astype(np.float64); yr = rankdata(y).astype(np.float64)
    xrc = xr - xr.mean(); yrc = yr - yr.mean()
    sxr = np.sqrt((xrc**2).sum()); syr = np.sqrt((yrc**2).sum())
    if sxr > 0 and syr > 0:
        yr_perms = np.array([rankdata(y_perms[k]) for k in range(n_perm)], dtype=np.float64)
        yrc_perms = yr_perms - yr_perms.mean(1, keepdims=True)
        syr_perms = np.sqrt((yrc_perms**2).sum(1))
        R['abs_spearman_rho'] = (abs(float((xrc*yrc).sum()/(sxr*syr))),
                                 np.abs((xrc * yrc_perms).sum(1) / (sxr * syr_perms)))
    else:
        R['abs_spearman_rho'] = (0., np.zeros(n_perm))

    # ── Slopes (raw + standardized) ──
    sort_idx = np.argsort(x)
    n1, n2 = n//3, 2*n//3
    seg_slices = {'overall': slice(None), 'early': slice(None, n1),
                  'mid': slice(n1, n2), 'late': slice(n2, None)}

    for pfx, sxs, syo, syp in [
        ('raw', x[sort_idx], y[sort_idx], y_perms[:, sort_idx]),
        ('std', None, None, None),
    ]:
        if pfx == 'std':
            xlo, xhi = x.min(), x.max()
            xn = (x - xlo) / (xhi - xlo) if xhi > xlo else np.full(n, 0.5)
            sxs = xn[sort_idx]
            ylo_p = y_perms.min(1, keepdims=True)
            yhi_p = y_perms.max(1, keepdims=True)
            yr_p = yhi_p - ylo_p
            syp = np.where(yr_p > 0, (y_perms - ylo_p) / yr_p, 0.5)[:, sort_idx]
            ylo_o, yhi_o = y.min(), y.max()
            syo = ((y - ylo_o) / (yhi_o - ylo_o) if yhi_o > ylo_o else np.full(n, 0.5))[sort_idx]

        ep_seg_obs, ep_seg_null = [], []
        for sname, slc in seg_slices.items():
            seg_x = sxs[slc]; seg_yo = syo[slc]; seg_yp = syp[:, slc]
            if len(seg_x) < 2:
                R[f'abs_{pfx}_ep_{sname}'] = (0., np.zeros(n_perm))
                R[f'abs_{pfx}_pf_{sname}'] = (0., np.zeros(n_perm))
                continue
            dx = seg_x[-1] - seg_x[0]
            if abs(dx) > 0:
                ep_o = abs(float((seg_yo[-1] - seg_yo[0]) / dx))
                ep_n = np.abs((seg_yp[:, -1] - seg_yp[:, 0]) / dx)
            else:
                ep_o = 0.; ep_n = np.zeros(n_perm)
            R[f'abs_{pfx}_ep_{sname}'] = (ep_o, ep_n)
            seg_xc = seg_x - seg_x.mean()
            denom_pf = (seg_xc**2).sum()
            if denom_pf > 0 and len(seg_x) >= 3:
                pf_o = abs(float((seg_xc * (seg_yo - seg_yo.mean())).sum() / denom_pf))
                pf_n = np.abs((seg_xc * (seg_yp - seg_yp.mean(1, keepdims=True))).sum(1) / denom_pf)
            else:
                pf_o = 0.; pf_n = np.zeros(n_perm)
            R[f'abs_{pfx}_pf_{sname}'] = (pf_o, pf_n)
            if sname != 'overall':
                ep_seg_obs.append(ep_o); ep_seg_null.append(ep_n)
        if ep_seg_null:
            R[f'{pfx}_seg_strength'] = (float(np.mean(ep_seg_obs)), np.mean(ep_seg_null, axis=0))

    # ── Bin metrics (equal-width + equal-count) ──
    for bt, abbr in [('equal_width', 'ew'), ('equal_count', 'ec')]:
        bi = _precompute_bins(x, bin_type=bt)
        y_mean = float(y.mean())
        ss_tot = float(((y - y_mean)**2).sum())

        if bi is None or ss_tot <= 0:
            for nm in [f'{abbr}_bin_eta2', f'{abbr}_bin_amp',
                       f'{abbr}_bin_bw_mean', f'{abbr}_bin_bw_early',
                       f'{abbr}_bin_bw_mid', f'{abbr}_bin_bw_late']:
                R[nm] = (0., np.zeros(n_perm))
            continue

        B_ind, bcounts = bi
        nb = len(bcounts)
        bm_obs = (B_ind @ y) / bcounts
        bm_null = (y_perms @ B_ind.T) / bcounts

        R[f'{abbr}_bin_eta2'] = (
            float((bcounts * (bm_obs - y_mean)**2).sum() / ss_tot),
            (bcounts * (bm_null - y_mean)**2).sum(1) / ss_tot)
        R[f'{abbr}_bin_amp'] = (
            float(bm_obs.max() - bm_obs.min()),
            bm_null.max(1) - bm_null.min(1))

        masks_bool = [B_ind[b].astype(bool) for b in range(nb)]
        bw_obs_arr = np.empty(nb)
        bw_null_arr = np.empty((n_perm, nb))
        for b in range(nb):
            m = masks_bool[b]
            bw_obs_arr[b] = np.percentile(y[m], 95) - np.percentile(y[m], 5)
            yb_n = y_perms[:, m]
            bw_null_arr[:, b] = np.percentile(yb_n, 95, axis=1) - np.percentile(yb_n, 5, axis=1)

        R[f'{abbr}_bin_bw_mean'] = (float(bw_obs_arr.mean()), bw_null_arr.mean(1))
        i1b, i2b = nb // 3, 2 * nb // 3
        for nm, slc in [(f'{abbr}_bin_bw_early', slice(None, max(i1b, 1))),
                        (f'{abbr}_bin_bw_mid', slice(max(i1b, 1), max(i2b, i1b + 1))),
                        (f'{abbr}_bin_bw_late', slice(max(i2b, i1b + 1), None))]:
            o = float(bw_obs_arr[slc].mean()) if len(bw_obs_arr[slc]) else 0.
            n_ = bw_null_arr[:, slc].mean(1) if bw_null_arr[:, slc].shape[1] > 0 else np.zeros(n_perm)
            R[nm] = (o, n_)

    return R

In [6]:
PHASE1_METRICS = None
phase1_obs_all = {}
phase1_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase1_summaries = []

t0 = time.time()
for i in tqdm(range(n_cases), desc='Phase 1'):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)
    R = compute_phase1_case(x, y, perms)

    if PHASE1_METRICS is None:
        PHASE1_METRICS = sorted(R.keys())
        for nm in PHASE1_METRICS:
            phase1_obs_all[nm] = np.empty(n_cases)

    row = {}
    z_nulls = []
    for nm in PHASE1_METRICS:
        obs_val, null_arr = R[nm]
        phase1_obs_all[nm][i] = obs_val
        d = -1 if nm in LOWER_IS_SIGNAL else 1
        z_o, z_n, med, iqr, p = _z_and_p(obs_val, null_arr.astype(np.float64), d)
        z_nulls.append(z_n)
        row[f'{nm}_obs'] = obs_val
        row[f'{nm}_null_med'] = med
        row[f'{nm}_null_iqr'] = iqr
        row[f'z_{nm}'] = z_o
        row[f'p_{nm}'] = p

    phase1_max_z_null[i] = np.stack(z_nulls).max(axis=0).astype(np.float32)
    phase1_summaries.append(row)

    if (i + 1) % 10000 == 0:
        el = time.time() - t0; rate = (i + 1) / el
        print(f'  {i+1:>7,}/{n_cases:,}  ({rate:.0f}/s, ETA {(n_cases-i-1)/rate/60:.0f}min)')

elapsed = time.time() - t0
print(f'Phase 1 done: {n_cases:,} cases, {len(PHASE1_METRICS)} metrics, {elapsed/60:.1f} min')
np.savez_compressed(CKPT_DIR / 'phase1.npz', max_z_null=phase1_max_z_null)
print(f'Metrics: {PHASE1_METRICS}')

Phase 1:  59%|█████▉    | 10001/17008 [20:35<08:30, 13.72it/s] 

   10,000/17,008  (8/s, ETA 14min)


Phase 1: 100%|██████████| 17008/17008 [28:59<00:00,  9.78it/s]


Phase 1 done: 17,008 cases, 33 metrics, 29.0 min
Metrics: ['abs_covariance', 'abs_pearson_r', 'abs_raw_ep_early', 'abs_raw_ep_late', 'abs_raw_ep_mid', 'abs_raw_ep_overall', 'abs_raw_pf_early', 'abs_raw_pf_late', 'abs_raw_pf_mid', 'abs_raw_pf_overall', 'abs_spearman_rho', 'abs_std_ep_early', 'abs_std_ep_late', 'abs_std_ep_mid', 'abs_std_ep_overall', 'abs_std_pf_early', 'abs_std_pf_late', 'abs_std_pf_mid', 'abs_std_pf_overall', 'ec_bin_amp', 'ec_bin_bw_early', 'ec_bin_bw_late', 'ec_bin_bw_mean', 'ec_bin_bw_mid', 'ec_bin_eta2', 'ew_bin_amp', 'ew_bin_bw_early', 'ew_bin_bw_late', 'ew_bin_bw_mean', 'ew_bin_bw_mid', 'ew_bin_eta2', 'raw_seg_strength', 'std_seg_strength']


## Phase 2: Distance Covariance

In [7]:
phase2_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase2_summaries = []

t0 = time.time()
for i in tqdm(range(n_cases), desc='Phase 2 (dcov)'):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    A = _double_center(x)
    B = _double_center(y)

    dcov_obs = np.sqrt(max(float((A * B).mean()), 0))
    dcor_xx = np.sqrt(max(float((A * A).mean()), 0))
    dcor_yy = np.sqrt(max(float((B * B).mean()), 0))
    dcor_obs = dcov_obs / np.sqrt(dcor_xx * dcor_yy) if dcor_xx > 0 and dcor_yy > 0 else 0.

    dcov_null = np.empty(N_PERM)
    dcor_null = np.empty(N_PERM)
    for k in range(N_PERM):
        p = perms[k]
        Bp = B[p][:, p]
        dcv = np.sqrt(max(float((A * Bp).mean()), 0))
        dcov_null[k] = dcv
        dcr_yy_p = np.sqrt(max(float((Bp * Bp).mean()), 0))
        dcor_null[k] = dcv / np.sqrt(dcor_xx * dcr_yy_p) if dcor_xx > 0 and dcr_yy_p > 0 else 0.

    row = {}
    z_nulls = []
    for nm, obs, null in [('dcov', dcov_obs, dcov_null), ('dcor', dcor_obs, dcor_null)]:
        z_o, z_n, med, iqr, pv = _z_and_p(obs, null.astype(np.float64), 1)
        z_nulls.append(z_n)
        row[f'{nm}_obs'] = obs
        row[f'{nm}_null_med'] = med
        row[f'{nm}_null_iqr'] = iqr
        row[f'z_{nm}'] = z_o
        row[f'p_{nm}'] = pv

    phase2_max_z_null[i] = np.stack(z_nulls).max(0).astype(np.float32)
    phase2_summaries.append(row)

    if (i + 1) % 5000 == 0:
        el = time.time() - t0; rate = (i + 1) / el
        print(f'  {i+1:>7,}/{n_cases:,}  ({rate:.1f}/s, ETA {(n_cases-i-1)/rate/3600:.1f}h)')

elapsed = time.time() - t0
print(f'Phase 2 done: {elapsed/3600:.1f}h')
np.savez_compressed(CKPT_DIR / 'phase2.npz', max_z_null=phase2_max_z_null)

Phase 2 (dcov):  29%|██▉       | 5002/17008 [08:25<16:32, 12.09it/s]

    5,000/17,008  (9.9/s, ETA 0.3h)


Phase 2 (dcov):  59%|█████▉    | 10001/17008 [16:44<10:44, 10.88it/s]

   10,000/17,008  (10.0/s, ETA 0.2h)


Phase 2 (dcov):  88%|████████▊ | 15002/17008 [24:29<02:44, 12.17it/s]

   15,000/17,008  (10.2/s, ETA 0.1h)


Phase 2 (dcov): 100%|██████████| 17008/17008 [27:32<00:00, 10.29it/s]


Phase 2 done: 0.5h


## Phase 3: Distribution Metrics (KS + Wasserstein)

In [8]:
phase3_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase3_summaries = []

t0 = time.time()
for i in tqdm(range(n_cases), desc='Phase 3 (dist)'):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)
    y_perms = y[perms]

    row = {}
    z_nulls = []

    for bt, abbr in [('equal_width', 'ew'), ('equal_count', 'ec')]:
        bi = _precompute_bins(x, bin_type=bt)
        if bi is None or len(bi[1]) < 3:
            for nm in [f'{abbr}_dist_ks', f'{abbr}_dist_wass']:
                row[f'{nm}_obs'] = 0.; row[f'{nm}_null_med'] = 0.
                row[f'{nm}_null_iqr'] = 0.; row[f'z_{nm}'] = 0.; row[f'p_{nm}'] = 1.
                z_nulls.append(np.zeros(N_PERM))
            continue

        B_ind, bcounts = bi
        nv = len(bcounts)
        low_mask = B_ind[:max(nv // 3, 1)].max(0).astype(bool)
        high_mask = B_ind[max(2 * nv // 3, nv // 3 + 1):].max(0).astype(bool)

        y_lo_o, y_hi_o = y[low_mask], y[high_mask]
        if len(y_lo_o) < 2 or len(y_hi_o) < 2:
            for nm in [f'{abbr}_dist_ks', f'{abbr}_dist_wass']:
                row[f'{nm}_obs'] = 0.; row[f'{nm}_null_med'] = 0.
                row[f'{nm}_null_iqr'] = 0.; row[f'z_{nm}'] = 0.; row[f'p_{nm}'] = 1.
                z_nulls.append(np.zeros(N_PERM))
            continue

        ks_obs = float(ks_2samp(y_lo_o, y_hi_o).statistic)
        w_obs = float(wasserstein_distance(y_lo_o, y_hi_o))
        ks_null = np.empty(N_PERM); w_null = np.empty(N_PERM)
        for k in range(N_PERM):
            yl = y_perms[k, low_mask]; yh = y_perms[k, high_mask]
            ks_null[k] = ks_2samp(yl, yh).statistic
            w_null[k] = wasserstein_distance(yl, yh)

        for nm, obs, null in [(f'{abbr}_dist_ks', ks_obs, ks_null),
                               (f'{abbr}_dist_wass', w_obs, w_null)]:
            z_o, z_n, med, iqr, pv = _z_and_p(obs, null, 1)
            z_nulls.append(z_n)
            row[f'{nm}_obs'] = obs; row[f'{nm}_null_med'] = med
            row[f'{nm}_null_iqr'] = iqr; row[f'z_{nm}'] = z_o; row[f'p_{nm}'] = pv

    phase3_max_z_null[i] = np.stack(z_nulls).max(0).astype(np.float32) if z_nulls else 0.
    phase3_summaries.append(row)

    if (i + 1) % 5000 == 0:
        el = time.time() - t0; rate = (i + 1) / el
        print(f'  {i+1:>7,}/{n_cases:,}  ({rate:.1f}/s, ETA {(n_cases-i-1)/rate/3600:.1f}h)')

elapsed = time.time() - t0
print(f'Phase 3 done: {elapsed/3600:.1f}h')
np.savez_compressed(CKPT_DIR / 'phase3.npz', max_z_null=phase3_max_z_null)

Phase 3 (dist):  29%|██▉       | 5002/17008 [05:04<11:20, 17.63it/s]

    5,000/17,008  (16.5/s, ETA 0.2h)


Phase 3 (dist):  59%|█████▉    | 10001/17008 [10:08<07:12, 16.20it/s]

   10,000/17,008  (16.4/s, ETA 0.1h)


Phase 3 (dist):  88%|████████▊ | 15003/17008 [15:02<01:53, 17.70it/s]

   15,000/17,008  (16.6/s, ETA 0.0h)


Phase 3 (dist): 100%|██████████| 17008/17008 [16:59<00:00, 16.69it/s]


Phase 3 done: 0.3h


## Phase 4: MINE (Subset Mode)

In [9]:
n_sub = len(subset_idx)

def compute_mine(x, y):
    mine = MINEObj(alpha=0.6, c=15)
    mine.compute_score(x, y)
    return mine.mic(), mine.mas(), mine.mev(), mine.mcn()


if HAS_MINEPY:
    phase4_max_z_null = np.full((n_cases, N_PERM), np.nan, dtype=np.float32)
    phase4_summaries = {}

    t0 = time.time()
    for j in tqdm(range(n_sub), desc='Phase 4 (MINE)'):
        i = subset_idx[j]
        x = x_all[i].astype(np.float64)
        y = y_all[i].astype(np.float64)
        perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

        mic_o, mas_o, mev_o, mcn_o = compute_mine(x, y)
        mic_null = np.empty(N_PERM); mas_null = np.empty(N_PERM)
        mev_null = np.empty(N_PERM); mcn_null = np.empty(N_PERM)

        for k in range(N_PERM):
            mic_null[k], mas_null[k], mev_null[k], mcn_null[k] = compute_mine(x, y[perms[k]])

        row = {}
        z_nulls = []
        for nm, obs, null in [('mic', mic_o, mic_null), ('mas', mas_o, mas_null),
                               ('mev', mev_o, mev_null), ('mcn', mcn_o, mcn_null)]:
            z_o, z_n, med, iqr, pv = _z_and_p(obs, null, 1)
            z_nulls.append(z_n)
            row[f'{nm}_obs'] = obs; row[f'{nm}_null_med'] = med
            row[f'{nm}_null_iqr'] = iqr; row[f'z_{nm}'] = z_o; row[f'p_{nm}'] = pv

        phase4_max_z_null[i] = np.stack(z_nulls).max(0).astype(np.float32)
        phase4_summaries[i] = row

        if (j + 1) % 500 == 0:
            el = time.time() - t0; rate = (j + 1) / el
            np.savez_compressed(CKPT_DIR / 'phase4_partial.npz', max_z_null=phase4_max_z_null)
            print(f'  {j+1:>6,}/{n_sub:,}  ({rate:.1f}/s, ETA {(n_sub-j-1)/rate/3600:.1f}h)  [checkpoint saved]')

    elapsed = time.time() - t0
    print(f'Phase 4 done: {n_sub:,} cases, {elapsed/3600:.1f}h')
    np.savez_compressed(CKPT_DIR / 'phase4.npz', max_z_null=phase4_max_z_null)
else:
    phase4_max_z_null = None
    phase4_summaries = {}
    print('Phase 4 skipped (minepy not available)')

Phase 4 (MINE):   6%|▌         | 500/9008 [10:05<4:37:55,  1.96s/it]

     500/9,008  (0.8/s, ETA 2.9h)  [checkpoint saved]


Phase 4 (MINE):  11%|█         | 1000/9008 [19:55<2:30:27,  1.13s/it]

   1,000/9,008  (0.8/s, ETA 2.7h)  [checkpoint saved]


Phase 4 (MINE):  17%|█▋        | 1500/9008 [29:31<2:26:27,  1.17s/it]

...

   8,500/9,008  (0.9/s, ETA 0.2h)  [checkpoint saved]


Phase 4 (MINE): 100%|█████████▉| 9000/9008 [2:56:31<00:10,  1.33s/it]

   9,000/9,008  (0.8/s, ETA 0.0h)  [checkpoint saved]


Phase 4 (MINE): 100%|██████████| 9008/9008 [2:56:41<00:00,  1.18s/it]

Phase 4 done: 9,008 cases, 2.9h


In [19]:
## Phase 4.1: MIC-minus-r² only

def pearson_r2_safe(x, y):
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    r = np.corrcoef(x, y)[0, 1]
    return float(r ** 2) if np.isfinite(r) else 0.0


def compute_mic_minus_r2(x, y):
    mine = MINEObj(alpha=0.6, c=15)
    mine.compute_score(x, y)

    mic = float(mine.mic())
    r2 = pearson_r2_safe(x, y)

    return mic - r2


if HAS_MINEPY:
    phase4_1_max_z_null = np.full((n_cases, N_PERM), np.nan, dtype=np.float32)
    phase4_1_summaries = {}

    t0 = time.time()

    for j in tqdm(range(n_sub), desc='Phase 4.1 (MIC - r²)'):
        i = subset_idx[j]

        x = x_all[i].astype(np.float64)
        y = y_all[i].astype(np.float64)
        perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

        obs = compute_mic_minus_r2(x, y)

        null_arr = np.empty(N_PERM)
        for k in range(N_PERM):
            null_arr[k] = compute_mic_minus_r2(x, y[perms[k]])

        z_o, z_n, med, iqr, pv = _z_and_p(obs, null_arr, 1)

        phase4_1_max_z_null[i] = z_n.astype(np.float32)
        phase4_1_summaries[i] = {
            'mic_minus_r2_obs': obs,
            'mic_minus_r2_null_med': med,
            'mic_minus_r2_null_iqr': iqr,
            'z_mic_minus_r2': z_o,
            'p_mic_minus_r2': pv,
        }

        if (j + 1) % 500 == 0:
            elapsed = time.time() - t0
            rate = (j + 1) / elapsed
            np.savez_compressed(
                CKPT_DIR / 'phase4_1_partial.npz',
                max_z_null=phase4_1_max_z_null,
            )
            print(
                f'  {j+1:>6,}/{n_sub:,}  '
                f'({rate:.1f}/s, ETA {(n_sub-j-1)/rate/3600:.1f}h)  '
                '[checkpoint saved]'
            )

    elapsed = time.time() - t0
    print(f'Phase 4.1 done: {n_sub:,} cases, {elapsed/3600:.1f}h')

    np.savez_compressed(
        CKPT_DIR / 'phase4_1.npz',
        max_z_null=phase4_1_max_z_null,
    )

else:
    phase4_1_max_z_null = None
    phase4_1_summaries = {}
    print('Phase 4.1 skipped because minepy is not available.')

Phase 4.1 (MIC - r²):   0%|          | 0/9008 [00:00<?, ?it/s]

Phase 4.1 (MIC - r²):   6%|▌         | 500/9008 [09:38<2:39:50,  1.13s/it]

     500/9,008  (0.9/s, ETA 2.7h)  [checkpoint saved]


Phase 4.1 (MIC - r²):  11%|█         | 1000/9008 [19:34<3:05:18,  1.39s/it]

   1,000/9,008  (0.9/s, ETA 2.6h)  [checkpoint saved]


...

   8,500/9,008  (0.9/s, ETA 0.2h)  [checkpoint saved]


Phase 4.1 (MIC - r²): 100%|█████████▉| 9000/9008 [2:53:46<00:09,  1.18s/it]

   9,000/9,008  (0.9/s, ETA 0.0h)  [checkpoint saved]


Phase 4.1 (MIC - r²): 100%|██████████| 9008/9008 [2:53:55<00:00,  1.16s/it]

Phase 4.1 done: 9,008 cases, 2.9h


## Phase 5: LOWESS R² (Subset Mode)

In [10]:
LOWESS_FRAC = 0.3
LOWESS_IT = 3

def lowess_r2(x, y):
    if len(x) < 5:
        return 0.
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = sm_lowess(ys, xs, frac=LOWESS_FRAC, it=LOWESS_IT, return_sorted=True)
    yp = np.interp(xs, fitted[:, 0], fitted[:, 1])
    ss_res = float(((ys - yp)**2).sum())
    ss_tot = float(((ys - ys.mean())**2).sum())
    return 1. - ss_res / ss_tot if ss_tot > 0 else 0.


phase5_max_z_null = np.full((n_cases, N_PERM), np.nan, dtype=np.float32)
phase5_summaries = {}

t0 = time.time()
for j in tqdm(range(n_sub), desc='Phase 5 (LOWESS R²)'):
    i = subset_idx[j] 
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    obs = lowess_r2(x, y)
    null_arr = np.empty(N_PERM)
    for k in range(N_PERM):
        null_arr[k] = lowess_r2(x, y[perms[k]])

    z_o, z_n, med, iqr, pv = _z_and_p(obs, null_arr, 1)
    phase5_max_z_null[i] = z_n.astype(np.float32)
    phase5_summaries[i] = {
        'lowess_r2_obs': obs, 'lowess_r2_null_med': med,
        'lowess_r2_null_iqr': iqr, 'z_lowess_r2': z_o, 'p_lowess_r2': pv
    }

    if (j + 1) % 500 == 0:
        el = time.time() - t0; rate = (j + 1) / el
        np.savez_compressed(CKPT_DIR / 'phase5_partial.npz', max_z_null=phase5_max_z_null)
        print(f'  {j+1:>6,}/{n_sub:,}  ({rate:.1f}/s, ETA {(n_sub-j-1)/rate/3600:.1f}h)  [checkpoint saved]')

elapsed = time.time() - t0
print(f'Phase 5 done: {n_sub:,} cases, {elapsed/3600:.1f}h')
np.savez_compressed(CKPT_DIR / 'phase5.npz', max_z_null=phase5_max_z_null)

Phase 5 (LOWESS R²):   6%|▌         | 500/9008 [11:48<3:09:13,  1.33s/it]

     500/9,008  (0.7/s, ETA 3.3h)  [checkpoint saved]


Phase 5 (LOWESS R²):  11%|█         | 1000/9008 [22:57<2:58:50,  1.34s/it]

   1,000/9,008  (0.7/s, ETA 3.1h)  [checkpoint saved]


Phase 5 (LOWESS R²):  17%|█▋        | 1500/9008 [34:04<2:45:22,  1.32s/it]

...

   8,500/9,008  (0.7/s, ETA 0.2h)  [checkpoint saved]


Phase 5 (LOWESS R²): 100%|█████████▉| 9000/9008 [3:28:32<00:10,  1.33s/it]

   9,000/9,008  (0.7/s, ETA 0.0h)  [checkpoint saved]


Phase 5 (LOWESS R²): 100%|██████████| 9008/9008 [3:28:42<00:00,  1.39s/it]

Phase 5 done: 9,008 cases, 3.5h


## Joint Test and Classification

In [20]:
# Merge all phase summaries
all_rows = []
for i in range(n_cases):
    row = {'case_id': int(cases_df['case_id'].iloc[i]),
           'source': cases_df['source'].iloc[i]}
    row.update(phase1_summaries[i])
    row.update(phase2_summaries[i])
    row.update(phase3_summaries[i])
    if i in phase4_summaries:
        row.update(phase4_summaries[i])
    if i in phase4_1_summaries:
        row.update(phase4_1_summaries[i])
    if i in phase5_summaries:
        row.update(phase5_summaries[i])
    all_rows.append(row)

# Joint T from max-Z across all phases
T_null_joint = np.maximum(phase1_max_z_null, phase2_max_z_null)
T_null_joint = np.maximum(T_null_joint, phase3_max_z_null)
if phase4_max_z_null is not None:
    p4 = phase4_max_z_null.copy()
    p4[np.isnan(p4)] = -np.inf
    T_null_joint = np.maximum(T_null_joint, p4)
if phase4_1_max_z_null is not None:
    p4_1 = phase4_1_max_z_null.copy()
    p4_1[np.isnan(p4_1)] = -np.inf
    T_null_joint = np.maximum(T_null_joint, p4_1)
p5 = phase5_max_z_null.copy()
p5[np.isnan(p5)] = -np.inf
T_null_joint = np.maximum(T_null_joint, p5)

# Collect all z_ columns for T_obs
z_cols = [c for c in all_rows[0] if c.startswith('z_')]
for i, row in enumerate(all_rows):
    T_obs = max((row.get(c, -np.inf) for c in z_cols), default=0.)
    p_val = float(np.sum(T_null_joint[i] >= T_obs) + 1) / (N_PERM + 1)
    row['T_joint'] = T_obs
    row['p_value'] = p_val

print(f'Assembly done: {len(z_cols)} z-score metrics in joint test')

Assembly done: 39 z-score metrics in joint test


In [21]:
df = pd.DataFrame(all_rows)
df['classification'] = 'uncertain'
df.loc[df['p_value'] <= 0.05, 'classification'] = 'detectable'
df.loc[df['p_value'] >= 0.10, 'classification'] = 'not_detectable'

out_path = OUT_DIR / 'permutation_all.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved {out_path}  ({len(df):,} rows × {len(df.columns)} cols)')
print()
print(df['classification'].value_counts().to_string())
print()
print(df.head(3))

Saved output/S3/permutation_all.parquet  (17,008 rows × 230 cols)

classification
detectable        12778
not_detectable     3883
uncertain           347

   case_id source  abs_covariance_obs  abs_covariance_null_med  \
0        3   main            0.002000                 0.000460   
1        4   main            0.002833                 0.000511   
2        7   main            0.005607                 0.000675   

   abs_covariance_null_iqr  z_abs_covariance  p_abs_covariance  \
0                 0.000467          3.295589          0.009901   
1                 0.000650          3.574117          0.009901   
2                 0.000820          6.012454          0.009901   

   abs_pearson_r_obs  abs_pearson_r_null_med  abs_pearson_r_null_iqr  ...  \
0           0.141068                0.032464                0.032954  ...   
1           0.188816                0.034037                0.043305  ...   
2           0.244736                0.029445                0.035807  ...   

   mic